# 13 — Feature Engineering for Clustering (Objective 1)

**Objective.** Decide whether any feature *derived* from existing columns should join the four clustering inputs,
then fit the scaler on the final set and save the model-ready input.

This section applies *"creating derived variables from existing fields and dropping those that add no explanatory
value."* Both halves matter: a derived feature is proposed for a business reason, then kept only if the evidence says
it adds something.

**Starting point:** 24,092 rated warehouses; inputs `product_wg_ton`, `num_refill_req_l3m`,
`transport_issue_l1y`, `wh_breakdown_l3m`; no encoding, no shape transformation, `StandardScaler` chosen.

**Input:** `Obj_1_Clustering/data/processed/clustering_base.csv`; `data/preprocessed/warehouse_preprocessed.csv`
(for one candidate that needs a condition column).
**Output:** `data/processed/clustering_input_scaled.csv` · `feature_engine/standard_scaler.pkl` ·
`feature_engine/candidate_features.csv` · an updated `feature_engine/feature_spec.md`.

## 0. Setup

In [ ]:
import sys, pathlib

ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
            if (p / "src" / "common.py").exists())
sys.path.insert(0, str(ROOT))
from src.common import *

set_style()

paths = obj_paths(1)
base = pd.read_csv(paths["processed"] / "clustering_base.csv")
inputs = ["product_wg_ton", "num_refill_req_l3m", "transport_issue_l1y", "wh_breakdown_l3m"]

print(f"clustering_base.csv: {base.shape[0]:,} rows x {base.shape[1]} columns")
print("current inputs:", inputs)

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import joblib
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler

---
## 1. What a derived feature has to show

The tests are fixed **before** any candidate is measured, so the results cannot shape the rules. Each rests on a
decision already taken with evidence:

| # | Test | Why this test | Anchor from earlier evidence |
|---|---|---|---|
| T1 | **Business meaning** — the feature describes warehouse *performance* in terms a manager would recognise | segments must be readable in the profiling step | the project objective |
| T2 | **New information** — the four current inputs cannot already predict it. Measured by R² of a linear regression of the candidate on the four inputs; **R² of 0.90 or more** marks a near-copy | a near-copy counts the same thing twice in the distance calculation — the reason storage issues was removed | Storage issues were removed as a near-duplicate of shipment weight (r = 0.987, R² = 0.974) |
| T3 | **No extremes beyond those already accepted** — measured by the furthest warehouse from the mean and the share beyond 3 standard deviations | a feature with stronger extremes would need a shape transform, which the earlier analysis declined | The earlier analysis found the four inputs reach at most 3.51 SD, with at most 1.39% of warehouses beyond 3 SD |
| T4 | **Performance only** — no condition column enters the segment definition | segments are defined by performance; conditions explain them afterwards | Segments are defined by performance; conditions are used only for profiling |

A candidate must pass all four.

---
## 2. Candidates, and the business reason for each

| Candidate | Formula | Business reason |
|---|---|---|
| `problems_total` | `transport_issue_l1y` + `wh_breakdown_l3m` | one headline count of reported problems, as a dashboard would show |
| `transport_issue_3m` | `transport_issue_l1y` ÷ 4 | transport issues are counted over **one year**, every other measure over **three months** (data dictionary: *l1y* vs *l3m*). Dividing by four puts all measures on the same window. |
| `problems_per_1000t` | (`transport_issue_l1y` ÷ 4 + `wh_breakdown_l3m`) ÷ (`product_wg_ton` ÷ 1,000) | **problem intensity** — problems per 1,000 t shipped over three months. A warehouse shipping heavily with few problems is operating efficiently; this puts that in one number. |
| `refills_per_1000t` | `num_refill_req_l3m` ÷ (`product_wg_ton` ÷ 1,000) | **refill intensity** — refills relative to volume shipped |
| `tons_per_worker` | `product_wg_ton` ÷ `workers_num` | **labour productivity** — shipment per worker |

`tons_per_worker` is included deliberately although it divides by a condition (`workers_num`), because productivity per
worker is the first efficiency measure many managers would ask about; T4 decides it on the record rather than leaving it
untested. It also inherits the 990 worker counts filled in the data cleaning step.

A ratio involving `storage_issue_reported_l3m` is not proposed: storage issues were moved to profiling as a near-copy of
shipment weight, and a ratio of the two would measure only the small part in which they differ — the component that
the EDA found holds 0.24% of the variance.

In [ ]:
workers = load_preprocessed().set_index("Ware_house_ID")["workers_num"]

candidates = pd.DataFrame({
    "problems_total": base["transport_issue_l1y"] + base["wh_breakdown_l3m"],
    "transport_issue_3m": base["transport_issue_l1y"] / 4,
    "problems_per_1000t": (base["transport_issue_l1y"] / 4 + base["wh_breakdown_l3m"])
                          / (base["product_wg_ton"] / 1000),
    "refills_per_1000t": base["num_refill_req_l3m"] / (base["product_wg_ton"] / 1000),
    "tons_per_worker": base["product_wg_ton"] / base["Ware_house_ID"].map(workers).to_numpy(),
})

assert candidates.notna().all().all() and np.isfinite(candidates.to_numpy()).all()
candidates.describe().T.round(3)

> **Interpretation.**
>
> - All five candidates were computed for every one of the 24,092 rated warehouses, with no gaps and
>   no division-by-zero values — the assertion checks both. The ranges already hint at what §3 will measure:
>
> - `problems_total` runs from 1 to 11 and `transport_issue_3m` from 0 to 1.25 — simple re-expressions of existing
>   inputs.
> - The two per-1,000 t ratios have long right tails: `problems_per_1000t` has a median of 0.172 but a maximum of
>   1.665, and `refills_per_1000t` a median of 0.181 but a maximum of 1.973.
> - `tons_per_worker` runs from 70 to 3,934 t per worker, around a median of 781.


---
## 3. Measuring each candidate against T2 and T3

For every candidate, over the 24,092 rated warehouses:

- **`r2_from_4_inputs`** (T2) — how much of the candidate a straight-line combination of the four inputs already
  reproduces.
- **`most_related_input`** — the input it correlates with most strongly, and that correlation.
- **`skew`, `furthest_sd`, `pct_beyond_3_sd`** (T3) — compare with the 3.51 SD and 1.39% accepted for the transport-issue extremes.
- **`mean_shipment_top_1pct`** — the average shipment weight of the warehouses with the highest 1% of the candidate's
  values, against 22,731 t for all rated warehouses. For a ratio with shipment weight underneath, this shows whether
  its extreme values come from a genuinely high numerator or simply from a small denominator.

Two extra checks, one for each candidate that raises a specific question:

- **`transport_issue_3m`** — after standard scaling, is it any different from `transport_issue_l1y`?
- **`tons_per_worker`** (T4) — how strongly does it correlate with the condition column it is built from?

In [ ]:
rows = []
for name in candidates.columns:
    v = candidates[name]
    z = (v - v.mean()) / v.std(ddof=0)
    corr = base[inputs].corrwith(v)
    top = corr.abs().idxmax()
    rows.append({
        "candidate": name,
        "r2_from_4_inputs": round(LinearRegression().fit(base[inputs], v).score(base[inputs], v), 4),
        "most_related_input": top,
        "corr_with_it": round(corr[top], 3),
        "skew": round(v.skew(), 3),
        "furthest_sd": round(z.abs().max(), 2),
        "pct_beyond_3_sd": round(100 * (z.abs() > 3).mean(), 2),
        "mean_shipment_top_1pct": round(base.loc[v >= v.quantile(0.99), "product_wg_ton"].mean()),
    })
evaluation = pd.DataFrame(rows)

scaled_transport = StandardScaler().fit_transform(base[["transport_issue_l1y"]])
scaled_3m = StandardScaler().fit_transform(candidates[["transport_issue_3m"]])
print(f"transport_issue_3m vs transport_issue_l1y after standard scaling — largest difference: "
      f"{np.abs(scaled_transport - scaled_3m).max():.2e}")
worker_corr = candidates["tons_per_worker"].corr(base["Ware_house_ID"].map(workers))
print(f"tons_per_worker — correlation with workers_num (a condition): {worker_corr:.3f}")
print(f"mean shipment of all rated warehouses: {base['product_wg_ton'].mean():,.0f} t\n")

evaluation

> **Interpretation.**
>
> Reading across the table: **two candidates fail T2 immediately.** `problems_total` (R² 1.0000) and
> `transport_issue_3m` (R² 1.0000) are exactly reproduced by the four current inputs — adding either would
> double-count those measures. `transport_issue_3m` makes this concrete: after standard scaling it is identical
> to `transport_issue_l1y` (largest difference 0.00), because dividing a column by four changes its units but not
> its shape, and the scaler removes units.
>
> **Three candidates pass T2 but fail T3.** `problems_per_1000t` reaches **8.83 standard deviations** from the
> mean, and `refills_per_1000t` reaches **6.17** — both well beyond the 3.51 SD and 1.39% accepted for the
> current inputs. `tons_per_worker` reaches 6.28 SD and also fails T4: it correlates at −0.431 with `workers_num`,
> a condition rather than a performance measure. The histograms below extend the picture for the three ratio and
> productivity candidates.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))
for ax, c in zip(axes, ["problems_per_1000t", "refills_per_1000t", "tons_per_worker"]):
    sns.histplot(candidates[c], bins=60, ax=ax)
    ax.set_title(c, fontsize=10); ax.set_xlabel(""); ax.set_ylabel("")
fig.suptitle("The three ratio candidates — rated warehouses", fontweight="bold", y=1.03)
plt.tight_layout(); plt.show()

> **Interpretation.**
>
> - Every candidate passes **T1** by construction — each was proposed for a stated business reason.
>   Against the other three tests:
>
> | Candidate | T2 — new information | T3 — extremes | T4 — performance only | Result |
> |---|---|---|---|---|
> | `problems_total` | **fails**: R² 1.0000 | passes: 3.32 SD, 0.18% | passes | not added |
> | `transport_issue_3m` | **fails**: R² 1.0000; identical after scaling | passes: 3.51 SD, 1.39% | passes | not added |
> | `problems_per_1000t` | passes: R² 0.6742 | **fails**: 8.83 SD, 2.08% | passes | not added |
> | `refills_per_1000t` | passes: R² 0.6690 | **fails**: 6.17 SD, 2.83% | passes | not added |
> | `tons_per_worker` | passes: R² 0.7365 | **fails**: 6.28 SD | **fails**: r = −0.431 with `workers_num` | not added |
>
> - **The two re-expressions add nothing.** `problems_total` is exactly transport issues plus breakdowns, so the four
>   inputs reproduce it perfectly (R² 1.0000). Adding it would count those two measures a second time — the
>   double-counting avoided. `transport_issue_3m` is the more instructive case: **after standard scaling it is
>   identical to `transport_issue_l1y`** (largest difference 0.00). Dividing a column by four changes its units but not
>   its shape, and standard scaling removes units. The one-year versus three-month mismatch is real, but the scaler
>   chosen earlier has already neutralised it.
>
> - **The two volume ratios fail on extremes, and the last column shows why.** Their most extreme warehouses lie 8.83
>   and 6.17 standard deviations from the mean — far beyond the 3.51 accepted for transport issues. The warehouses in their top 1%
>   ship on average only **5,925 t** and **4,925 t**, against 22,731 t for all rated warehouses. So their extreme values
>   come from a small number *below* the line rather than a large number above it: a low-volume warehouse with an
>   ordinary number of problems or refills produces a very large ratio. Read from the histograms, both have a tall peak
>   at low values and a long, thin right tail, and `refills_per_1000t` has its tallest bar at zero, where warehouses with
>   no refill requests sit. As inputs, these ratios would pull segments toward isolating a few very small warehouses, and
>   taming them would require the log transform that was declined earlier.
>
> - **Labour productivity fails twice.** `tons_per_worker` reaches 6.28 standard deviations, and its top 1% ship on
>   average **45,708 t**: its extremes are the heaviest-shipping warehouses, so it largely re-describes shipment weight
>   (r = 0.858). It also correlates at −0.431 with `workers_num`, a condition, so using it would bring staffing into the
>   definition of segments, contrary to the rule that segments are defined by performance measures only. The histogram shows a broad hump with a right tail reaching about 3,900 t
>   per worker.
>
> - **None of the five passes all four tests.**


---
## 4. The final clustering inputs

> **Decision — candidate features.**
>
> - Keep the clustering inputs as:
>   - `product_wg_ton`;
>   - `num_refill_req_l3m`;
>   - `transport_issue_l1y`;
>   - `wh_breakdown_l3m`.
> - Reject `problems_total`: fully reproduced by the current inputs (R² 1.0000).
> - Reject `transport_issue_3m`: fully reproduced by the current inputs and identical after scaling (R² 1.0000).
> - Reject `problems_per_1000t`: extreme values are driven by low-volume warehouses (8.83 SD from the mean).
> - Reject `refills_per_1000t`: extreme values are driven by low-volume warehouses (6.17 SD from the mean).
> - Reject `tons_per_worker`: too extreme for a clustering input and built on a condition column (T3 and T4).
>
> **What is still kept for business interpretation.**
>
> - Efficiency remains visible because shipment weight and problem counts are separate inputs.
> - The profiling step can report total problems per 1,000 t at segment level.
> - The profiling step can report tons per worker at segment level.
> - Segment-level ratios avoid the small-denominator distortion that ruled the ratios out as inputs.
>
> **Record kept.**
>
> - All five candidate variables were created.
> - All five were measured against fixed tests.
> - All five were dropped on evidence.
> - The record is saved in `feature_engine/candidate_features.csv`.

In [ ]:
final_inputs = inputs      # set by the decision above

pd.DataFrame({"decision": ["kept" if c in final_inputs else "not added" for c in candidates.columns]},
             index=candidates.columns).join(evaluation.set_index("candidate")[["r2_from_4_inputs", "furthest_sd"]])

> **Interpretation.**
>
> - The table restates the feature decision alongside the two figures that decided it: all five candidates are
>   marked "not added", and `final_inputs` is the original four columns, which the rest of the notebook uses.

---
## 5. Fitting the scaler

`StandardScaler` is fitted on the final inputs over all 24,092 rated warehouses — no train/test split is used,
because clustering has no held-out answers to protect.

In [ ]:
scaler = StandardScaler().fit(base[final_inputs])

scaled = pd.DataFrame(scaler.transform(base[final_inputs]), columns=final_inputs)
scaled.insert(0, "Ware_house_ID", base["Ware_house_ID"])

print("fitted parameters — what the scaler subtracts and divides by:")
print(pd.DataFrame({"mean_subtracted": scaler.mean_, "std_divided_by": scaler.scale_},
                   index=final_inputs).round(3).to_string())
print("\nafter scaling:")
scaled[final_inputs].describe().T[["mean", "std", "min", "max"]].round(3)

> **Interpretation.**
>
> - The fitted scaler subtracts each input's mean over the rated warehouses — 22,730.988 t,
>   4.093 refills, 0.779 transport issues and 3.613 breakdowns — and divides by each
>   standard deviation: 11,343.255 t, 2.607, 1.203 and 1.578. The scaler was fitted on
>   exactly the intended rows.
>
> - After scaling, every input has mean 0 and standard deviation 1, and the ranges match the earlier preview
>   exactly — for example, transport issues from −0.648 to 3.509.
>
> - **One consequence for reading results.** A scaled value of +1 means about 11,343 t above the mean for shipment weight,
>   but about 1.6 breakdowns above the mean for breakdowns. The model building step will produce cluster centres in these
>   standardised units, which is why the fitted scaler is saved: the profiling step converts centres back into tons and
>   counts before describing any segment.

---
## 6. Save

- **`clustering_input_scaled.csv`** — `Ware_house_ID` and the scaled inputs. The input to the model building step.
- **`standard_scaler.pkl`** — the fitted scaler, so scaled values can be turned back into recorded units when cluster
  centres are read in the profiling step.
- **`candidate_features.csv`** — the evaluation table, as the record of what was tested and why each candidate was
  or was not added.
- **`feature_spec.md`** — updated with this notebook's decision.

In [ ]:
save_table(scaled, paths["processed"] / "clustering_input_scaled.csv", index=False)

scaler_path = paths["feature_engine"] / "standard_scaler.pkl"
joblib.dump(scaler, scaler_path)
print(f"saved  {scaler_path.relative_to(PROJECT_ROOT)}")

save_table(evaluation.assign(added=evaluation["candidate"].isin(final_inputs)),
           paths["feature_engine"] / "candidate_features.csv", index=False)

spec_path = paths["feature_engine"] / "feature_spec.md"
spec = spec_path.read_text(encoding="utf-8").split("\n## Feature engineering")[0].rstrip() + "\n"
rows_md = "\n".join(
    f"| `{r.candidate}` | {r.r2_from_4_inputs} | {r.furthest_sd} | {r.pct_beyond_3_sd}% | "
    f"{'added' if r.candidate in final_inputs else 'not added'} |"
    for r in evaluation.itertuples())
spec += f'''
## Feature engineering

Five derived candidates were tested against four fixed tests (business meaning; R² from the current inputs below 0.90;
extremes no stronger than the 3.51 SD / 1.39% accepted in the earlier step; no condition column in the segment definition).

| Candidate | R² from the 4 inputs | Furthest SD | Beyond 3 SD | Result |
|---|---|---|---|---|
{rows_md}

**Final clustering inputs ({len(final_inputs)}):** {", ".join(f"`{c}`" for c in final_inputs)}.

**Model-ready file:** `data/processed/clustering_input_scaled.csv` ({scaled.shape[0]:,} rows), scaled with
`feature_engine/standard_scaler.pkl` (`StandardScaler`, fitted on all rated warehouses).
'''
spec_path.write_text(spec, encoding="utf-8")
print(f"saved  {spec_path.relative_to(PROJECT_ROOT)}")

---
## 7. Checks

The saved files are read back. The scaler is reloaded from disk and applied to the unscaled inputs, and must reproduce
the saved scaled file exactly — proving NB 15 can use it to translate cluster centres back into tons and counts.

In [ ]:
back = pd.read_csv(paths["processed"] / "clustering_input_scaled.csv")
reloaded = joblib.load(paths["feature_engine"] / "standard_scaler.pkl")

assert back.shape == (24_092, 1 + len(final_inputs)), back.shape
assert list(back.columns) == ["Ware_house_ID"] + final_inputs
assert back.isna().sum().sum() == 0
assert back["Ware_house_ID"].equals(base["Ware_house_ID"])
assert np.allclose(back[final_inputs].mean(), 0, atol=1e-9)
assert np.allclose(back[final_inputs].std(ddof=0), 1, atol=1e-9)
assert np.allclose(reloaded.transform(base[final_inputs]), back[final_inputs])
assert np.allclose(reloaded.inverse_transform(back[final_inputs]), base[final_inputs])

print("all checks passed")
print(f"clustering_input_scaled.csv : {back.shape[0]:,} rows x {back.shape[1]} columns; every input mean 0, sd 1")
print("standard_scaler.pkl         : reproduces the scaled file and converts it back to recorded units")

---
## Summary

**Feature engineering tested five derived features and added none.**

| Candidate | Business idea | Why not added |
|---|---|---|
| `problems_total` | a headline count of problems | R² 1.0000 from the inputs — it would double-count |
| `transport_issue_3m` | put transport issues on the three-month window | identical to `transport_issue_l1y` after standard scaling |
| `problems_per_1000t` | problem intensity | extremes to 8.83 SD, from low-volume warehouses (top 1% ship 5,925 t) |
| `refills_per_1000t` | refill intensity | extremes to 6.17 SD, from low-volume warehouses (top 1% ship 4,925 t) |
| `tons_per_worker` | labour productivity | extremes to 6.28 SD; built on a condition (r = −0.431 with `workers_num`) |

**The four original inputs — no derived feature added.**

- The four performance measures are the final clustering inputs:
  - `product_wg_ton`;
  - `num_refill_req_l3m`;
  - `transport_issue_l1y`;
  - `wh_breakdown_l3m`.

**Output.**

- `data/processed/clustering_input_scaled.csv` — 24,092 rated warehouses; key plus four standardised inputs, each with
  mean 0 and standard deviation 1. **The input to the model building step.**
- `feature_engine/standard_scaler.pkl` — reloaded from disk and verified to reproduce the scaled file and to convert it
  back into recorded units.
- `feature_engine/candidate_features.csv` and an updated `feature_engine/feature_spec.md`.

**For the profiling step.**

- Report efficiency per segment as total problems / total tons.
- Report tons per worker at segment level.
- Use segment totals so the low-volume denominator problem does not distort the interpretation.

**Input to the model building step.**

- Four equally weighted inputs for 24,092 warehouses.
- Separation scores are expected to be modest because the performance data is a continuum.